# Cross-chain master vault CCTP backtest

Backtest notebook for the Ethereum primary-chain vault-of-vaults strategy with Base and Arbitrum satellite vaults. The notebook uses real vault history and generated CCTP bridge pairs, then asserts that CCTP trades are generated and the run finishes profitably.


## Notebook setup

Set up the Trading Strategy client, chart output, and imports used throughout the notebook.


In [ ]:
import datetime
import importlib.util
import logging
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from dotenv import load_dotenv
from eth_defi.token import USDC_NATIVE_TOKEN
from tradingstrategy.chain import ChainId
from tradingstrategy.client import Client

from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.execution_context import notebook_execution_context
from tradeexecutor.strategy.parameters import StrategyParameters
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.strategy.reserve_currency import ReserveCurrency
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.utils.notebook import OutputMode, display_head_and_tail, setup_charting_and_output
from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns

load_dotenv(override=True)
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=900)

api_key = os.environ.get("TRADING_STRATEGY_API_KEY")
client = Client.create_jupyter_client(api_key=api_key)
logger = logging.getLogger("strategy")


## Strategy module

Load the production `xchain-master-vault.py` strategy and narrow its denomination filter to native USDC, because CCTP mints native USDC on destination chains.


In [ ]:
repo_root = Path.cwd()
if not (repo_root / "strategies" / "xchain-master-vault.py").exists():
    repo_root = Path.cwd().parents[1]
strategy_path = repo_root / "strategies" / "xchain-master-vault.py"
spec = importlib.util.spec_from_file_location("xchain_master_vault", strategy_path)
xchain_master_vault = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(xchain_master_vault)

# CCTP V2 bridges native USDC. Keep the backtest universe to native-USDC vaults
# so bridge-funded satellite deposits spend the token that CCTP actually mints.
xchain_master_vault.ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}


## Universe and parameters

Use Ethereum as the primary reserve chain and Base/Arbitrum as CCTP satellite chains. The vault list is the production hand-curated list narrowed to real native-USDC vaults on these three chains.


In [ ]:
SUPPORTING_PAIRS = [
    (ChainId.ethereum, "uniswap-v3", "WETH", "USDC", 0.0005),
    (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
    (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
    (ChainId.base, "uniswap-v3", "WETH", "USDC", 0.0005),
]

SOURCE_VAULTS = [
    # Ethereum native-USDC vaults
    (ChainId.ethereum, "0x09c4c7b1d2e9aa7506db8b76f1dbbd61c08c114b"),
    (ChainId.ethereum, "0x438982ea288763370946625fd76c2508ee1fb229"),
    (ChainId.ethereum, "0x8df3deba711ae4a9af16cbca5e4fbb1402f036d5"),
    (ChainId.ethereum, "0xca790385506b790554571cbc9da73f0130cdcfd5"),
    (ChainId.ethereum, "0xe9d33286f0e37f517b1204aa6da085564414996d"),

    # Base native-USDC vaults
    (ChainId.base, "0xf7e26fa48a568b8b0038e104dfd8abdf0f99074f"),
    (ChainId.base, "0x3094b241aade60f91f1c82b0628a10d9501462f9"),
    (ChainId.base, "0x70fffbacb53ef74903ac074aae769414a70970d1"),
    (ChainId.base, "0x3ec4a293fb906dd2cd440c20decb250def141df1"),
    (ChainId.base, "0x8092ca384d44260ea4feaf7457b629b8dc6f88f0"),
    (ChainId.base, "0xc777031d50f632083be7080e51e390709062263e"),
    (ChainId.base, "0xad20523a7dc37babc1cc74897e4977232b3d02e5"),
    (ChainId.base, "0xbc10718571fcb3c3f67800e7c0887e450d2ff398"),
    (ChainId.base, "0xefe32813dba3a783059d50e5358b9e3661218dad"),
    (ChainId.base, "0xd5c22fa3f7ee979ed7c28e36669b29797ab277e4"),

    # Arbitrum native-USDC vaults
    (ChainId.arbitrum, "0x75288264fdfea8ce68e6d852696ab1ce2f3e5004"),
    (ChainId.arbitrum, "0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a"),
    (ChainId.arbitrum, "0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c"),
    (ChainId.arbitrum, "0x1723cb57af58efb35a013870c90fcc3d60174a4e"),
    (ChainId.arbitrum, "0x20d419a8e12c45f88fda7c5760bb6923cee27f98"),
    (ChainId.arbitrum, "0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0"),
    (ChainId.arbitrum, "0x0df2e3a0b5997adc69f8768e495fd98a4d00f134"),
]


class Parameters(xchain_master_vault.Parameters):
    """Notebook backtest parameters for Ethereum, Arbitrum, and Base."""

    id = "xchain-master-vault-eth-arb-base-cctp-notebook"
    primary_chain_id = ChainId.ethereum
    supporting_pairs = SUPPORTING_PAIRS
    source_vaults = SOURCE_VAULTS
    preferred_stablecoin = USDC_NATIVE_TOKEN[ChainId.ethereum].lower()
    auto_generate_cctp_bridges = True

    backtest_start = datetime.datetime(2025, 1, 1)
    backtest_end = datetime.datetime(2026, 3, 11)
    initial_cash = 100_000

    max_assets_in_portfolio = 12
    allocation_pct = 0.98
    min_tvl_usd = 7_500
    individual_rebalance_min_threshold_usd = 50.0
    sell_rebalance_min_threshold_usd = 10.0


parameters = StrategyParameters.from_class(Parameters)


## Trading universe

Build the real-data trading universe and check that the synthetic CCTP bridge pairs were generated for both satellite chains.


In [ ]:
universe_input = CreateTradingUniverseInput(
    client=client,
    timestamp=None,
    parameters=parameters,
    execution_context=notebook_execution_context,
    execution_model=None,
    universe_options=UniverseOptions.from_strategy_parameters_class(Parameters, notebook_execution_context),
)
strategy_universe = xchain_master_vault.create_trading_universe(universe_input)

pairs = list(strategy_universe.iterate_pairs())
vault_pairs = [pair for pair in pairs if pair.kind.is_vault()]
bridge_pairs = [pair for pair in pairs if pair.kind.is_cctp_bridge()]

universe_summary = pd.DataFrame(
    [
        ("Chains", ", ".join(sorted(chain.get_name() for chain in strategy_universe.data_universe.chains))),
        ("Total pairs", len(pairs)),
        ("Vault pairs", len(vault_pairs)),
        ("CCTP bridge pairs", len(bridge_pairs)),
        ("Vault denominations", ", ".join(sorted({pair.quote.token_symbol for pair in vault_pairs}))),
        ("Bridge destinations", ", ".join(str(pair.get_destination_chain_id()) for pair in bridge_pairs)),
    ],
    columns=["Metric", "Value"],
)

display(universe_summary)

assert {pair.get_destination_chain_id() for pair in bridge_pairs} == {ChainId.arbitrum.value, ChainId.base.value}
assert {pair.quote.token_symbol for pair in vault_pairs} == {"USDC"}


## Backtest

Run the production decision logic on the narrowed three-chain universe. `max_workers=1` keeps notebook execution deterministic across local shells and CI-like environments.


In [ ]:
result = run_backtest_inline(
    client=client,
    universe=strategy_universe,
    decide_trades=xchain_master_vault.decide_trades,
    create_indicators=xchain_master_vault.create_indicators,
    parameters=Parameters,
    initial_deposit=Parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=Parameters.routing,
    engine_version=xchain_master_vault.trading_strategy_engine_version,
    max_workers=1,
    name=Parameters.id,
)

state = result.state
indicator_data = result.indicators
trades = list(state.portfolio.get_all_trades())
bridge_trades = [trade for trade in trades if trade.pair.is_cctp_bridge()]
vault_trades = [trade for trade in trades if trade.pair.kind.is_vault()]
equity_curve = calculate_equity_curve(state)
returns = calculate_returns(equity_curve)
final_equity = state.portfolio.get_total_equity()
total_return = final_equity / Parameters.initial_cash - 1

summary_df = pd.DataFrame(
    [
        ("Initial cash USD", Parameters.initial_cash),
        ("Final equity USD", final_equity),
        ("Total return", total_return),
        ("Equity curve points", len(equity_curve)),
        ("All trades", len(trades)),
        ("Vault trades", len(vault_trades)),
        ("CCTP bridge trades", len(bridge_trades)),
        ("Successful CCTP bridge trades", sum(trade.is_success() for trade in bridge_trades)),
    ],
    columns=["Metric", "Value"],
)

display(summary_df)

assert final_equity > Parameters.initial_cash, f"Expected profitable run, got final equity {final_equity:,.2f}"
assert len(bridge_trades) > 0, "Expected CCTP bridge trades to be generated"
assert all(trade.is_success() for trade in bridge_trades), "Expected all CCTP bridge trades to execute"


## CCTP trade diagnostics

Inspect bridge direction, destination chain, size, and status. These diagnostics are CCTP-specific and guard against silent bridge-planner regressions.


In [ ]:
def _bridge_direction(trade):
    return "bridge out" if trade.is_buy() else "bridge back"


bridge_rows = []
for trade in bridge_trades:
    bridge_rows.append(
        {
            "trade_id": trade.trade_id,
            "opened_at": trade.opened_at,
            "executed_at": trade.executed_at,
            "direction": _bridge_direction(trade),
            "source_chain_id": trade.pair.get_source_chain_id(),
            "destination_chain_id": trade.pair.get_destination_chain_id(),
            "value_usd": trade.get_value(),
            "quantity": float(trade.executed_quantity or 0),
            "status": trade.get_status().value,
            "position_id": trade.position_id,
        }
    )

bridge_df = pd.DataFrame(bridge_rows).sort_values(["opened_at", "trade_id"])
display_head_and_tail(bridge_df, head_rows=10, tail_rows=10)

bridge_summary_df = bridge_df.groupby(["destination_chain_id", "direction"]).agg(
    trades=("trade_id", "count"),
    value_usd=("value_usd", "sum"),
    first_trade=("opened_at", "min"),
    last_trade=("opened_at", "max"),
).reset_index()

display(bridge_summary_df)

assert set(bridge_df["destination_chain_id"]) == {ChainId.arbitrum.value, ChainId.base.value}
assert {"bridge out", "bridge back"}.issubset(set(bridge_df["direction"])), "Expected both bridge directions"
assert bridge_df["value_usd"].sum() > 0


## Bridge and vault sequencing

Check that cycles containing satellite vault trades also include CCTP bridge trades. This specifically exercises the PR #1453 bridge-trade injection path.


In [ ]:
trade_rows = []
for trade in trades:
    trade_rows.append(
        {
            "cycle": trade.strategy_cycle_at,
            "trade_id": trade.trade_id,
            "kind": trade.pair.kind.value,
            "chain_id": trade.pair.chain_id,
            "is_bridge": trade.pair.is_cctp_bridge(),
            "is_satellite_vault": trade.pair.kind.is_vault() and trade.pair.chain_id != Parameters.primary_chain_id.value,
            "value_usd": trade.get_value(),
            "status": trade.get_status().value,
        }
    )

trade_df = pd.DataFrame(trade_rows)
cycle_df = trade_df.groupby("cycle").agg(
    trades=("trade_id", "count"),
    bridge_trades=("is_bridge", "sum"),
    satellite_vault_trades=("is_satellite_vault", "sum"),
    traded_value_usd=("value_usd", "sum"),
).reset_index()

cycles_with_satellite_vaults = cycle_df[cycle_df["satellite_vault_trades"] > 0]
cycles_missing_bridges = cycles_with_satellite_vaults[cycles_with_satellite_vaults["bridge_trades"] == 0]

display_head_and_tail(cycle_df, head_rows=10, tail_rows=10)
display(cycles_missing_bridges)

assert len(cycles_with_satellite_vaults) > 0, "Expected satellite vault trades"
assert len(cycles_missing_bridges) == 0, "Every satellite-vault cycle should have CCTP bridge trades"


## Profit calculation check

Double-check the profit calculation directly from the equity curve and portfolio final equity.


In [ ]:
curve_start = float(equity_curve.iloc[0])
curve_end = float(equity_curve.iloc[-1])
curve_return = curve_end / curve_start - 1
portfolio_return = final_equity / Parameters.initial_cash - 1

profit_check_df = pd.DataFrame(
    [
        ("Equity curve start", curve_start),
        ("Equity curve end", curve_end),
        ("Equity curve return", curve_return),
        ("Portfolio final equity", final_equity),
        ("Portfolio return", portfolio_return),
        ("Absolute difference", abs(curve_end - final_equity)),
    ],
    columns=["Metric", "Value"],
)

display(profit_check_df)

assert abs(curve_end - final_equity) < 0.01
assert curve_return > 0
assert abs(curve_return - portfolio_return) < 0.0001


## Equity curve

Plot the resulting equity curve and daily return distribution.


In [ ]:
equity_df = equity_curve.rename("equity_usd").reset_index()
equity_df.columns = ["timestamp", "equity_usd"]
fig = px.line(equity_df, x="timestamp", y="equity_usd", title="Cross-chain master vault equity curve")
fig.show()

returns_df = returns.rename("return").reset_index()
returns_df.columns = ["timestamp", "return"]
fig = px.histogram(returns_df, x="return", nbins=50, title="Daily return distribution")
fig.show()


## Vault and chain diagnostics

Summarise final open exposure by chain and show the tail of the executed vault trades.


In [ ]:
position_rows = []
for position in state.portfolio.get_open_and_frozen_positions():
    if position.pair.is_cctp_bridge():
        continue
    position_rows.append(
        {
            "position_id": position.position_id,
            "chain_id": position.pair.chain_id,
            "pair": position.pair.get_ticker(),
            "value_usd": position.get_value(),
            "opened_at": position.opened_at,
        }
    )

positions_df = pd.DataFrame(position_rows)
chain_exposure_df = positions_df.groupby("chain_id").agg(
    positions=("position_id", "count"),
    value_usd=("value_usd", "sum"),
).reset_index()
chain_exposure_df["weight"] = chain_exposure_df["value_usd"] / chain_exposure_df["value_usd"].sum()

display(chain_exposure_df)
display_head_and_tail(positions_df.sort_values("value_usd", ascending=False), head_rows=10, tail_rows=5)


## Standard performance view

Render the standard chart registry from the production strategy for performance and allocation review.


In [ ]:
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.performance_metrics import performance_metrics
from tradeexecutor.strategy.chart.standard.weight import weight_allocation_statistics

charts = xchain_master_vault.create_charts(
    timestamp=None,
    parameters=parameters,
    strategy_universe=strategy_universe,
    execution_context=notebook_execution_context,
)
chart_renderer = ChartBacktestRenderingSetup(
    registry=charts,
    strategy_input_indicators=indicator_data,
    state=state,
    backtest_start_at=Parameters.backtest_start,
    backtest_end_at=Parameters.backtest_end,
)

display(chart_renderer.render(performance_metrics))
display(chart_renderer.render(weight_allocation_statistics))
display_head_and_tail(chart_renderer.render(xchain_master_vault.trading_pair_breakdown_with_chain), head_rows=10, tail_rows=10)
